# External Managed Tables

**Managed Table and External**

A table has Metadata
The Actual Records are persisted in the storage. 
Depending on how we create the table its life cycle is managed differenty

In [0]:
%sql
USE datax_db;
SHOW TABLES;

# WHAT IS MANAGED TABLE?

- Databricks/metastore manages the table metadata.
- We don't manually specify an external data location when creating it.
- The system determines the managed storage location.
- The table's data lifecycle is associated with the table.


In [0]:
%sql
Describe Extended datax_db.orders

In [0]:
managed_detail=spark.sql("""Describe Extended datax_db.orders""")
display(managed_detail)

In [0]:
managed_location = (    spark.sql("DESCRIBE DETAIL datax_db.orders")    .select("location")    .first()[0])
print(managed_location)

# INTRODUCING EXTERNAL TABLES

_The External Table says_

- Metadata registered in Metastore 
- Data is stored explicitly in another specified location 
- Storage lifecycle is controlled independenly from the table metadata

In [0]:
orders_df=spark.read.format("delta").load("/Volumes/dataxdevelopment/default/delta_datax")

In [0]:
orders_df=spark.read.format("delta").table("dataxdevelopment.default.orders_delta")
display(orders_df)

https://dataxazure.blob.core.windows.net/dev-datax/landing/processed_orders/2026/04/07/SampleData%20-%20orders_bad_20260406%20(1).csv

## Setting Up External Location for Azure Storage

To write data to an external Azure storage path, Unity Catalog needs:
1. **Storage Credential** — authenticates to the Azure storage account (uses an Access Connector managed identity)
2. **External Location** — maps a cloud storage path to that credential

> **Prerequisites:** Your Azure admin must grant the **Storage Blob Data Contributor** role on the `dataxazure` storage account to the Databricks Access Connector's managed identity.

In [0]:
%sql
-- Step 1: Create a storage credential using your Azure Access Connector
-- Replace <access-connector-resource-id> with your actual Access Connector resource ID
-- Format: /subscriptions/<sub-id>/resourceGroups/<rg>/providers/Microsoft.Databricks/accessConnectors/<connector-name>

SHOW STORAGE CREDENTIALS;



In [0]:
%sql
DESCRIBE STORAGE CREDENTIAL dataxazure_credential;

In [0]:
%sql
SELECT
    external_location_name,
    url,
    storage_credential_name,
    storage_credential_id,
    read_only
FROM system.information_schema.external_locations
WHERE external_location_name = 'dataxazure_landing';

In [0]:
# External location V2
orders_path = (
    "abfss://dev-datax@dataxazure.dfs.core.windows.net/"
    "landing_v2/processed_orders/"
)

# Write orders DataFrame as Delta
orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(orders_path)

print("Orders data written successfully!")

In [0]:
display(orders_df.limit(10))

In [0]:
path = "abfss://dev-datax@dataxazure.dfs.core.windows.net/landing/"

display(dbutils.fs.ls(path))

In [0]:
%sql
CREATE TABLE datax_db.orders_external 
USING DELTA 
LOCATION 'abfss://dev-datax@dataxazure.dfs.core.windows.net/landing_v2/processed_orders/'

In [0]:
%sql
DESCRIBE EXTENDED datax_db.orders_external 

# When Does managed table makes sense?

- Databricks owns/manages the table lifecycle 
- We don't need independent management of the physical data location.
- We want simpler table management.


# When Does External table makes sense?

- Data already exists at a controlled storage location
- Storage lifecyle must remain independent
- organization wants explicit control over where data resides.
